In [2]:
import torch
import torch.nn as nn

class MultiHeadAttention(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.context_length = config["context_length"]
        self.d_in = config["n_embd"]
        self.d_out = config["n_embd"]
        self.num_heads = config["n_heads"]
        self.dropout = nn.Dropout(config["dropout_rate"])
        self.qkv_bias = config["qkv_bias"]
        assert self.d_out % self.num_heads == 0, "n_embd must be divisible by n_heads"
        self.d_head = self.d_out // self.num_heads
        self.W_k = nn.Linear(self.d_in, self.d_out, bias=self.qkv_bias)
        self.W_q = nn.Linear(self.d_in, self.d_out, bias=self.qkv_bias)
        self.W_v = nn.Linear(self.d_in, self.d_out, bias=self.qkv_bias)
        self.mask = torch.tril(torch.ones(self.context_length, self.context_length))
        self.projection = nn.Linear(self.d_out, self.d_out)

    def forward(self, x):
        B, N, D = x.shape   # D is d_in, N  is context_length
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        Q = Q.view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        K = K.view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        V = V.view(B, N, self.num_heads, self.d_head).transpose(1, 2)
        QKT = Q @ K.transpose(2, 3)
        QKT = QKT.masked_fill(self.mask[:N, :N] == 0, float('-inf'))
        attention_probs = torch.softmax(QKT / (self.d_head ** 0.5), dim=-1)
        attention_probs = self.dropout(attention_probs)
        context_vector = attention_probs @ V
        context_vector = context_vector.transpose(1, 2).contiguous().view(B, N, self.d_out)
        return self.projection(context_vector)

class FeedForward(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config["n_embd"], 4 * config["n_embd"]),
            nn.GELU(),
            nn.Linear(4 * config["n_embd"], config["n_embd"])
        )

    def forward(self, x):
        return self.layers(x)

class LayerNorm(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(config["n_embd"]))
        self.beta = nn.Parameter(torch.zeros(config["n_embd"]))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        x = (x - mean) / (std + self.eps)
        x = self.gamma * x + self.beta
        return x

# LayerNorm(config) calls __init__ of LayerNorm and generates
# an instance of LayerNorm called ln1
# ln1(x) calls forward of ln1 with x as input
class TransformerBlock(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config)
        self.attn = MultiHeadAttention(config)
        self.dropout = nn.Dropout(config["dropout_rate"])
        self.ff = FeedForward(config)
        self.ln2 = LayerNorm(config)

    def forward(self, x):
        # Shortcut: x +
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x

class Simple_GPT(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config["vocab_size"], config["n_embd"])
        self.position_embedding = nn.Embedding(config["context_length"], config["n_embd"])
        self.dropout = nn.Dropout(config["dropout_rate"])
        self.blocks = nn.Sequential(*[TransformerBlock(config) for _ in range(config["n_layers"])])
        # f(*[2, 3, 5, 7]) means f(2, 3, 5, 7)
        self.ln_f = LayerNorm(config)
        self.prediction_layer = nn.Linear(config["n_embd"], config["vocab_size"])

    def forward(self, x):
        B, N = x.shape      # B is batch size, N is context length
        token_embeddings = self.token_embedding(x)  # [B, N, n_embd]
        position_embeddings = self.position_embedding(torch.arange(N))  # [N, n_embd]
        x = token_embeddings + position_embeddings  # [B, N, n_embd]
        x = self.dropout(x)
        x = self.blocks(x)  # [B, N, n_embd]
        x = self.ln_f(x)
        logits = self.prediction_layer(x)   # [B, N, vocab_size]
        return logits


In [3]:
config = {
    "vocab_size": 50257,
    "context_length": 1024,
    "n_embd": 768,
    "n_heads": 12,
    "n_layers": 12,
    "dropout_rate": 0.1,
    "qkv_bias": False
}

In [4]:
input = torch.randint(0, 50257, (1, 1024))
print(input.shape)
model = Simple_GPT(config)
output = model(input)
print(output.shape)

torch.Size([1, 1024])
torch.Size([1, 1024, 50257])


In [5]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")

Total number of parameters: 163059793


In [6]:
def generate_text_sample(model, idx, max_new_tokens, context_length):
    # max_new_tokens is the number of tokens we want to generate
    # idx is the array of indices in the current context
    # idx has size [batch_size, n_tokens]
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]     # Takes the latest context window
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]       #   last word in new context window
        # we want to keep batch and vocab dimension same
        probs = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probs, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)     # dim=1 for the context window
    return idx

In [7]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
start_token = "Hello, I am"
encoded = torch.tensor(tokenizer.encode(start_token)).unsqueeze(0)
print(encoded)

ModuleNotFoundError: No module named 'tiktoken'

In [ ]:
model.eval()
output = generate_text_sample(model, encoded, 60, config['context_length'])
print(output.shape)
decoded = tokenizer.decode(output[0].squeeze(0).tolist())
print(decoded)

torch.Size([1, 64])
Hello, I am Impl LAST album FAM charges Chau Dengε comedic gain optANIillus149Yeah Ramadan Citation heterosexual basicrones itching cow acknowledges antioxid complimentridden emotshi rhetorical cuts 120SpaceEngineers accommodateeches nominee stand observe Basics diagnosed metabolic Koucosystemslaught fussmodernulousJJINK smokes presumptive Interest Millennialsgilected radiBILL evid 143 unequiv Leaders
